# Lektion 17 - Skapa Lokala AI-Agenter med Foundry Local och Qwen

I denna anteckningsbok bygger du en **lokal ingenjörsassistent** som körs helt och hållet på din arbetsstation. Ingen molnbaserad inferens används alls. Assistenten kan:

1. **Anropa verktyg** via Qwens funktionsanrop genom Foundry Local.
2. **Lista och läsa filer** inuti en sandlåded projektmapp.
3. **Analysera kod** med enkla mått.
4. **Söka i dokumentation** med lokal RAG (Chroma).
5. **Använda en lokal MCP-server** (hoppar över på ett smidigt sätt om ingen är konfigurerad).

Agentkoden ser nästan identisk ut med molnlektionerna — enda skillnaden är att klientändpunkten flyttas från molnet till `localhost`.


## Installation

Innan du kör detta anteckningsblock:

1. **Installera Microsoft Foundry Local** (se [dokumentationen](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) för ditt operativsystem).
2. **Ladda ner och starta en Qwen-modell:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Installera Python-paketen nedan.

Allt körs lokalt. En dator med ~8 GB RAM är ett realistiskt minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Anslut till Foundry Local

`FoundryLocalManager` laddar ner modellen vid behov, startar den lokala tjänsten och ger oss en **OpenAI-kompatibel endpoint**. Vi pekar sedan den standard OpenAI SDK mot den. API-nyckeln är en lokal platshållare — inga molnuppgifter används.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokala verktyg (sandlådebegränsade filoperationer)

Vi skapar ett litet exempelprojekt på disken och definierar sedan verktyg som är begränsade till den projektroten. Sandlådekontrollen spelar roll även lokalt: ett verktyg som läser godtyckliga sökvägar körs med din användares behörigheter och kan beröra allt som du kan.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokal RAG med Chroma

Vi bäddar in en liten uppsättning dokumentationsutdrag i en lokal Chroma-samling. Chroma körs i processen och lagrar vektorer på disk — ingen server, ingen moln. Verktyget `search_docs` hämtar de mest relevanta utdragen för en förfrågan.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Verktygsanropsslingan

Nu registrerar vi verktygen med modellen med hjälp av OpenAI-verktygsschemat och kör den standardiserade verktygsanropsslingan — modellen begär ett verktyg, vi kör det lokalt, matar tillbaka resultatet och upprepar tills modellen ger ett slutgiltigt svar. Qwens pålitliga funktionsanrop är det som gör att detta fungerar på enheten.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokal MCP (Frivillig)

MCP är en transport, inte en molntjänst — en MCP-server kan köras som en lokal process över `stdio`. Cellen nedan visar hur du skulle ansluta till en lokal MCP-server om du har en konfigurerad (till exempel en filsystemserver). Den hoppar förbi smidigt när `LOCAL_MCP_COMMAND` inte är satt, så att anteckningsboken fortfarande körs från början till slut utan den.

Säkerhetsnotering: en lokal MCP-server körs med dina användarbehörigheter. Begränsa den till en projektsmapp och validera dess resultat innan du agerar på dem.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Sammanfattning

Du byggde en ingenjörsassistent som körs helt på din maskin:

- **Foundry Local** serverade en **Qwen**-modell bakom en OpenAI-kompatibel endpoint — så agentkoden matchar molnlektionerna.
- **Sandboxade verktyg** gav agenten filåtkomst och kodanalys utan att lämna en projektkatalog.
- **Chroma** levererade **lokal RAG** över dokumentationen.
- **Lokal MCP** visade hur man återanvänder MCP-ekosystemet offline.

Ingen molnbaserad inferens användes vid någon tidpunkt.

### Utmaning

Utöka den lokala agenten till att:

1. **Arbeta med flera MCP-servrar** — anslut en filsystemserver och en git-server och låt agenten välja mellan dem.
2. **Använd lokal minne** — spara en kort samtalshistorik till disk så att assistenten kommer ihåg tidigare turer över anteckningsboksomstarter.
3. **Stötta lokal multitagentorkestrering** — lägg till en andra lokal agent (t.ex. en granskare) och låt de två samarbeta om en uppgift.

I nästa lektion kommer du att lära dig hur man säkrar distribuerade AI-agenter.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
